In [1]:
import os
from google.colab import userdata

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')


!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge

import zipfile
with zipfile.ZipFile("challenges-in-representation-learning-facial-expression-recognition-challenge.zip", "r") as zip_ref:
    zip_ref.extractall("fer2013_data")

print("Dataset downloaded and extracted successfully!")



100% 285M/285M [00:01<00:00, 207MB/s]

Dataset downloaded and extracted successfully!


In [2]:
import os

from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [3]:

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class FERDataset(Dataset):
  def __init__(self, X, y):
      self.X = X.reset_index(drop=True)
      self.y = y.reset_index(drop=True)

  def __len__(self):
      return len(self.X)

  def __getitem__(self, idx):
      pixels = np.fromstring(self.X.iloc[idx], sep=' ', dtype=np.uint8)
      img = pixels.reshape(48, 48)
      img = torch.tensor(img, dtype=torch.float32).unsqueeze(0) / 255.0
      label = torch.tensor(self.y.iloc[idx], dtype=torch.long)
      return img, label


In [4]:
import pandas as pd

train_split = pd.read_csv('/content/drive/MyDrive/FER_project/data/train_split.csv')
val_split = pd.read_csv('/content/drive/MyDrive/FER_project/data/val_split.csv')


In [5]:
X_train = train_split["pixels"]
y_train = train_split["emotion"]

X_val = val_split["pixels"]
y_val = val_split["emotion"]







In [6]:
train_dataset = FERDataset(X_train, y_train)
val_dataset = FERDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [7]:
import torch
import torch.nn as nn


class AdvancedCNN(nn.Module):
    def __init__(self):
        super(AdvancedCNN, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.1),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.2),


            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [8]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AdvancedCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [9]:
!pip install wandb -q

In [10]:
import wandb

wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nmetr23 (nmetr23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [11]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)



In [14]:
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import numpy as np

def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            loss = criterion(outputs, y)

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average="macro")
    precision = precision_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")
    cm = confusion_matrix(all_labels, all_preds)

    return total_loss / len(loader), acc, f1, precision, recall, cm, all_labels, all_preds

In [13]:
lrs = [1e-3, 1e-2, 1e-1]
batch_sizes = [10, 32, 64]
optimizers = ["adam", "sgd"]


In [ ]:
import itertools
import torch
import wandb

for lr, batch_size, opt_name in itertools.product(lrs, batch_sizes, optimizers):

    run_name = f"AdvancedCNN_lr{lr}_bs{batch_size}_opt{opt_name}"

    wandb.init(
        project="FER-Challenge",
        name=run_name,
        reinit=True
    )

    wandb.config.update({
        "lr": lr,
        "batch_size": batch_size,
        "optimizer": opt_name,
        "model": "AdvancedCNN"
    })

    model = AdvancedCNN().to(device)

    if opt_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    criterion = torch.nn.CrossEntropyLoss()

    epochs = 10

    for epoch in range(epochs):

        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)

        val_loss, val_acc, val_f1, val_precision, val_recall, cm, y_true, y_pred = evaluate(
            model, val_loader, criterion, device
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "val_f1": val_f1,
            "val_precision": val_precision,
            "val_recall": val_recall
        })

        print(f"{run_name} | Epoch {epoch+1}: acc={val_acc:.4f}, f1={val_f1:.4f}")



    wandb.log({
        "final_accuracy": val_acc,
        "final_f1": val_f1,
        "final_precision": val_precision,
        "final_recall": val_recall
    })



    import seaborn as sns
    import matplotlib.pyplot as plt

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=False, cmap="Blues")

    wandb.log({"confusion_matrix": wandb.Image(plt)})

    plt.close()

    wandb.finish()

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


AdvancedCNN_lr0.001_bs10_optadam | Epoch 1: acc=0.3905, f1=0.2477


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


AdvancedCNN_lr0.001_bs10_optadam | Epoch 2: acc=0.4603, f1=0.3341


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


AdvancedCNN_lr0.001_bs10_optadam | Epoch 10: acc=0.5441, f1=0.4471


epoch,▁▂▃▃▄▅▆▆▇█
final_accuracy,▁
final_f1,▁
final_precision,▁
final_recall,▁
train_loss,█▅▄▃▃▂▂▂▁▁
val_accuracy,▁▄▅▆▆▅▇█▇█
val_f1,▁▄▅▆▅▆▇█▆█
val_loss,█▆▄▃▃▄▂▁▂▁
val_precision,▁▅▅▆▆▇▇▇██
+1,...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


AdvancedCNN_lr0.001_bs10_optsgd | Epoch 1: acc=0.2579, f1=0.0765
